In [ ]:
3.	Factor performance: In this exercise, you are asked to explore some classic issues from the empirical literature on stock returns and factor models.  While we have now covered a number of factor models in class (as well as the related, though more general, APT), this question focuses on the performance of the returns on some of the most well-known factor portfolios. This question considers size- and value-sorted portfolios, which are the basis of the Fama-French three-factor model.
The relevant data can be found in the file Factors.csv. For the purposes of this question, we will deal with six assets: four Fama-French portfolios (small-low, small-high, big-low, big-high), the market portfolio, and the 30-day Treasury bill. The four Fama-French portfolios are the corners of the 2x3 size/value portfolios found on Ken French's website, http://mba.tuck.dartmouth.edu/pages/faculty/ken.french/. "Big-high," for example, refers to a portfolio of large stocks with high book-to-market ratios (i.e., large value stocks). The market portfolio is the value-weighted portfolio of stocks listed on NYSE, AMEX and NASDAQ. The data set runs from July 1926 through December 2025.
(c)	Do the exercises described in parts i-ii for the whole sample, and also for two subsamples: 7/26-12/63, and 1/64-12/25.
i.	Estimate the vector of sample mean excess returns, and the covariance matrix of excess returns, for each of the samples. Use these estimates to compute two ex-post mean-variance efficient sets: one for portfolios not including the risk-free asset, and one including the risk-free asset. Plot the two sets on a graph with the standard deviation of excess returns on the horizontal axis and the mean excess return on the vertical axis, and indicate where each of the four Fama-French portfolios and the market portfolio lie. Calculate the Sharpe ratios of the tangency portfolio and the market portfolio.
ii.	Plot expected return against beta for each of the portfolios. Calculate alphas, and discuss your results.
(d)	In recent years there has been concern that the publicity given to value investing, and the creation of quantitative investment strategies, may alter the properties of value returns. One variant of this concern is that the excess return to value (the value premium) may disappear permanently as quantitative investors bid up the price of value stocks to efficient levels. Another variant is that the excess return to value may become less stable, as capital flows in and out of value stocks in response to shifting sentiment of end investors about quantitative value strategies. Some have even argued that such shifting sentiment may cause the excess return to value to display a pattern of short-term positive autocorrelation ("style momentum") and longer-term negative autocorrelation ("style reversal").
iii.	Plot a one-year moving average of the excess return to small value stocks over small growth stocks (small-high minus small-low, or "small HML") over the period 1/64-12/25. Compare the behavior of the plot in the two subperiods 1/64-12/94 and 1/95-12/25.
iv.	Calculate the mean, standard deviation, and Sharpe ratio of the excess return on the market portfolio over the Treasury bill, and the return on small HML, for each of the two subperiods.
v.	Aggregate the small HML return to the quarterly data frequency and plot its autocorrelation function out to 12 quarters (3 years), i.e. Corr(Rt, Rt+τ), for τ = 0, 1, …, 12, in each of the two subperiods.
vi.	What do your results suggest about the changing behavior of value returns in recent years? Do they support any of the concerns described above?
(e)	Now consider the cumulative returns and alphas to a factor portfolio proposed in a recent paper of mine,  the "duration" factor. The paper builds on earlier work by another Haas faculty member, Martin Lettau, who proposed that the value premium arises as a result of value firms generating their cash flows (earnings and dividends) at a shorter horizon than growth firms, and that shorter-term firms are perceived as riskier by investors.  My paper builds on Martin's by showing empirically that a portfolio of short-term minus long-term stocks has a strong and positive CAPM alpha, and that this explains the value premium.
Yo

#Q3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# Data Loading and Preparation
# =============================================================================
df = pd.read_csv('Factors.csv', index_col=0)

# The index is loaded as integers (e.g., 192607). Convert to string, then to datetime.
df.index = pd.to_datetime(df.index.astype(str), format='%Y%m')
df.index.name = 'Date'

# Calculate excess returns using 'Rf'
assets = ['Small-Low', 'Small-High', 'Big-Low', 'Big-High', 'Market']
for asset in assets:
    df[f'{asset}_ex'] = df[asset] - df['Rf']

excess_assets = [f'{asset}_ex' for asset in assets]

# =============================================================================
# PART 3(a): Mean-Variance Frontier and Alphas
# =============================================================================

# Define the periods
periods = {
    'Whole Sample': ('1926-07-01', '2025-12-31'),
    'Subsample 1': ('1926-07-01', '1963-12-31'),
    'Subsample 2': ('1964-01-01', '2025-12-31')
}

print("--- PART 3(a) RESULTS ---")

for period_name, (start_date, end_date) in periods.items():
    print(f"\n--- {period_name} ({start_date} to {end_date}) ---")

    # Slice data
    sub_df = df.loc[start_date:end_date]
    ex_rets = sub_df[excess_assets]

    # i. Mean and Covariance
    mu = ex_rets.mean().values
    Sigma = ex_rets.cov().values
    Sigma_inv = np.linalg.inv(Sigma)
    ones = np.ones(len(mu))

    print("Mean Excess Returns:")
    for i, asset in enumerate(assets):
        print(f"  {asset}: {mu[i]:.6f}")

    print("\nCovariance Matrix:")
    print(Sigma)

    # Frontier Math
    A = ones.T @ Sigma_inv @ ones
    B = ones.T @ Sigma_inv @ mu
    C = mu.T @ Sigma_inv @ mu
    Delta = A * C - B**2

    # Tangency Portfolio
    w_tangency = (Sigma_inv @ mu) / B
    mu_tangency = C / B
    var_tangency = C / (B**2)
    std_tangency = np.sqrt(var_tangency)
    sharpe_tangency = mu_tangency / std_tangency

    # Market Portfolio (Market is the last asset in the list)
    mu_market = mu[-1]
    var_market = Sigma[-1, -1]
    std_market = np.sqrt(var_market)
    sharpe_market = mu_market / std_market

    print(f"\nSharpe Ratio - Tangency Portfolio: {sharpe_tangency:.6f}")
    print(f"Sharpe Ratio - Market Portfolio:   {sharpe_market:.6f}")

    # Plot Frontiers
    # Adjusting grid range slightly to handle percentage scale if applicable
    mu_grid = np.linspace(min(-1.0, mu.min() - 1.0), max(3.0, mu.max() + 2.0), 100)
    std_risky_frontier = np.sqrt((A * mu_grid**2 - 2 * B * mu_grid + C) / Delta)
    std_cml = mu_grid / np.sqrt(C)

    valid_cml = mu_grid >= 0

    plt.figure(figsize=(10, 6))
    plt.plot(std_risky_frontier, mu_grid, label='Risky Assets Frontier', color='blue')
    plt.plot(std_cml[valid_cml], mu_grid[valid_cml], label='Frontier with RF Asset (CML)', color='green', linestyle='--')

    stds = np.sqrt(np.diag(Sigma))
    for i, asset in enumerate(assets):
        plt.scatter(stds[i], mu[i], marker='o', s=80, label=asset)
        plt.annotate(asset, (stds[i] + (stds.max()*0.02), mu[i]))

    plt.scatter(std_tangency, mu_tangency, marker='*', s=150, color='red', label='Tangency Portfolio')

    plt.title(f'Mean-Variance Frontier: {period_name}')
    plt.xlabel('Standard Deviation of Excess Returns')
    plt.ylabel('Mean Excess Return')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'Frontier_{period_name.replace(" ", "_")}.pdf')
    plt.close()

    # ii. Expected Return vs Beta and Alphas
    print("\nBetas and Alphas:")
    var_m = Sigma[-1, -1]
    betas = []
    alphas = []

    for i, asset in enumerate(assets):
        cov_im = Sigma[i, -1]
        beta_i = cov_im / var_m
        alpha_i = mu[i] - beta_i * mu_market
        betas.append(beta_i)
        alphas.append(alpha_i)
        print(f"  {asset:12s} - Beta: {beta_i:.4f}, Alpha: {alpha_i:.6f}")

    plt.figure(figsize=(10, 6))
    plt.scatter(betas, mu, color='blue')
    for i, asset in enumerate(assets):
        plt.annotate(asset, (betas[i] + 0.02, mu[i]))

    b_grid = np.linspace(0, max(betas) + 0.5, 10)
    sml = b_grid * mu_market
    plt.plot(b_grid, sml, label='Security Market Line (SML)', color='red', linestyle='--')

    plt.title(f'Expected Return vs Beta: {period_name}')
    plt.xlabel('Beta')
    plt.ylabel('Mean Excess Return')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'SML_{period_name.replace(" ", "_")}.pdf')
    plt.close()

# =============================================================================
# PART 3(b): Small HML Performance and Changing Behavior
# NOTE: This section was generated with the assistance of Gemini.
# =============================================================================

print("\n\n--- PART 3(b) RESULTS ---")
# Gemini assisted code start

# Calculate the Small HML factor (Small-High minus Small-Low)
df['Small_HML'] = df['Small-High'] - df['Small-Low']

subperiod_1 = slice('1964-01-01', '1994-12-31')
subperiod_2 = slice('1995-01-01', '2025-12-31')

# 3(b) i. 1-year moving average of excess return to Small HML
hml_full_period = df.loc['1964-01-01':'2025-12-31'].copy()
hml_full_period['Small_HML_1yr_MA'] = hml_full_period['Small_HML'].rolling(window=12).mean()

plt.figure(figsize=(12, 6))
plt.plot(hml_full_period.index, hml_full_period['Small_HML_1yr_MA'], label='1-Year MA of Small HML', color='purple')
plt.axvline(pd.to_datetime('1995-01-01'), color='black', linestyle='--', label='Subperiod Split (1995)')
plt.title('1-Year Moving Average of Small HML Return (1964 - 2025)')
plt.xlabel('Date')
plt.ylabel('Moving Average of Return')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Small_HML_1yr_MA.pdf')
plt.close()

# 3(b) ii. Summary statistics
def calc_stats(series):
    """Calculates mean, std, and annualized Sharpe for a given series."""
    mean_ret = series.mean()
    std_ret = series.std()
    # Annualizing assuming monthly data
    sharpe = (mean_ret / std_ret) * np.sqrt(12)
    return mean_ret, std_ret, sharpe

for name, period_slice in zip(['1964-1994', '1995-2025'], [subperiod_1, subperiod_2]):
    print(f"\nStats for {name}:")
    period_data = df.loc[period_slice]

    mkt_mean, mkt_std, mkt_sharpe = calc_stats(period_data['Market_ex'])
    hml_mean, hml_std, hml_sharpe = calc_stats(period_data['Small_HML'])

    print(f"  Market Excess -> Mean: {mkt_mean:.6f}, Std: {mkt_std:.6f}, Annualized Sharpe: {mkt_sharpe:.4f}")
    print(f"  Small HML     -> Mean: {hml_mean:.6f}, Std: {hml_std:.6f}, Annualized Sharpe: {hml_sharpe:.4f}")

# 3(b) iii. Quarterly ACF of Small HML
plt.figure(figsize=(14, 6))

for idx, (name, period_slice) in enumerate(zip(['1964-1994', '1995-2025'], [subperiod_1, subperiod_2])):
    period_data = df.loc[period_slice]

    # Aggregate monthly returns to quarterly.
    # If the data is in percentages (e.g., 5.0 for 5%), we divide by 100 before compounding, then multiply back.
    # We'll use a safer generalized compounding method:
    qtr_hml = (1 + period_data['Small_HML']/100).resample('Q').prod() - 1

    lags = 12
    acf_vals = [qtr_hml.autocorr(lag=lag) for lag in range(lags + 1)]

    plt.subplot(1, 2, idx + 1)
    plt.bar(range(lags + 1), acf_vals, color='teal' if idx==0 else 'coral', alpha=0.7)
    plt.axhline(0, color='black', linewidth=1)
    plt.title(f'Quarterly Small HML ACF ({name})')
    plt.xlabel('Lags (Quarters)')
    plt.ylabel('Autocorrelation')
    plt.xticks(range(lags + 1))
    plt.ylim(-0.4, 1.0)
    plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('Small_HML_ACF_Quarterly.pdf')
plt.close()
# Gemini assisted code end

print("\nAll tasks completed. Plots saved as PDFs.")

FileNotFoundError: [Errno 2] No such file or directory: 'Factors.csv'